## 09_lenormand_multilevel
  Repeats the comparison at several spatial levels (CZ: obec, ORP, okres, kraj; PL: gmina,
  powiat, NUTS3): unit tables, distance and flow matrices, calibrated exponents, CPC.
  Output: data/interim/09_lenormand_multilevel/, data/processed/09_lenormand_multilevel/.
  Environment: nuts3_tdlm.

In [ ]:
import os
from pathlib import Path

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)
import pandas as pd

# --- config: all names, paths and levels in one place ---
INTERIM = Path("data/interim")
OUT_DIR = Path("data/processed/09_lenormand_multilevel")
OUT_FILE = OUT_DIR / "levels_summary.csv"
AREA_COL = "AREA_KM2"

CZ_INPUTS = INTERIM / "cz_obec_inputs.parquet"
CZ_OD = INTERIM / "cz_od_work_obec.parquet"
CZ_UNIT_ID = "LAU_ID"
CZ_KRAJ_PREFIX_LEN = 5            # kraj = first 5 characters of the okres code
PL_INPUTS = INTERIM / "pl_gmina_inputs.parquet"
PL_MAP = INTERIM / "pl_gmina_nuts3.csv"
PL_CODE_LEN = 6                   # gmina code length (leading zeros)

# --- CZ: obec -> ORP -> okres -> kraj mapping taken from the OD file ---
od = pd.read_parquet(CZ_OD)
cols = ["obec", "orp", "okres"]
orig = od[["op_obec_kod", "op_orp_kod", "op_lau1_kod"]].set_axis(cols, axis=1)
dest = od[["doj_obec_kod", "doj_orp_kod", "doj_lau1_kod"]].set_axis(cols, axis=1)
cz_map = pd.concat([orig, dest]).astype(str).drop_duplicates()
assert cz_map["obec"].is_unique, "obec code maps to more than one ORP/okres"
cz_map["kraj"] = cz_map["okres"].str[:CZ_KRAJ_PREFIX_LEN]

cz = pd.read_parquet(CZ_INPUTS)
cz["obec"] = cz[CZ_UNIT_ID].astype(str)
cz = cz.merge(cz_map, on="obec", how="left")
print("CZ obec without mapping:", int(cz["okres"].isna().sum()))

# --- PL: gmina -> powiat -> NUTS3 mapping ---
pl = pd.read_parquet(PL_INPUTS)
pl["gmina"] = pl["p6"].astype(str).str.zfill(PL_CODE_LEN)
pl_map = pd.read_csv(PL_MAP, dtype=str)[["gmina_code", "powiat_code", "nuts3_code"]]
pl_map.columns = ["gmina", "powiat", "nuts3"]
pl = pl.merge(pl_map, on="gmina", how="left")
print("PL gmina without mapping:", int(pl["powiat"].isna().sum()))

# --- summary per country and level ---
rows = []
for country, df, levels in [("CZ", cz, ["obec", "orp", "okres", "kraj"]),
                            ("PL", pl, ["gmina", "powiat", "nuts3"])]:
    for lvl in levels:
        area = df.dropna(subset=[lvl]).groupby(lvl)[AREA_COL].sum()
        n = len(area)
        rows.append({"country": country, "level": lvl, "n_units": n,
                     "ordered_pairs": n * (n - 1),
                     "mean_unit_area_km2": round(area.mean(), 1)})
summary = pd.DataFrame(rows)

OUT_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT_FILE, index=False)
print(summary.to_string(index=False))
print("saved:", OUT_FILE)

In [ ]:
# PROJ data fix: must run BEFORE geopandas / pyproj are imported
import os
import sys
from pathlib import Path
from importlib.util import find_spec


def find_proj_dir():
    """Return the first directory that contains proj.db (no pyproj import needed)."""
    candidates = [Path(sys.prefix) / "share" / "proj"]  # conda environment
    spec = find_spec("pyproj")
    if spec and spec.submodule_search_locations:  # pip wheel with bundled data
        candidates.append(Path(list(spec.submodule_search_locations)[0]) / "proj_dir" / "share" / "proj")
    for c in candidates:
        if (c / "proj.db").exists():
            return c
    return None


proj_dir = find_proj_dir()
if proj_dir is not None:
    os.environ["PROJ_LIB"] = str(proj_dir)
    os.environ["PROJ_DATA"] = str(proj_dir)
    print("PROJ data: proj.db found, environment variables set")
else:
    print("PROJ data: proj.db NOT found in the expected places")

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
INTERIM = Path("data/interim")
OUT_DIR = INTERIM / "09_lenormand_multilevel"
POP_COL, AREA_COL = "POP_2021", "AREA_KM2"
CZ_LEVELS = ["obec", "orp", "okres", "kraj"]
PL_LEVELS = ["gmina", "powiat", "nuts3"]
CZ_KRAJ_PREFIX_LEN = 5            # kraj = first 5 characters of the okres code
PL_CODE_LEN = 6                   # gmina code length (leading zeros)
CZ_INPUTS = INTERIM / "cz_obec_inputs.parquet"
CZ_OD = INTERIM / "cz_od_work_obec.parquet"
PL_INPUTS = INTERIM / "pl_gmina_inputs.parquet"
PL_MAP = INTERIM / "pl_gmina_nuts3.csv"
GEOM = Path("data/raw/gisco/LAU_RG_01M_2021_3035.geojson")   # raw GISCO file, correctly labelled EPSG:3035
CRS_METRIC = 3035
# reference distance tables of notebooks 06 / 06b (used only for the check)
REF = {("CZ", "kraj"): INTERIM / "lenormand_comparison/cz/distance_km.csv",
       ("PL", "nuts3"): INTERIM / "lenormand_comparison/pl/distance_km.csv"}
# ------------------------------------------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = lambda country, level: OUT_DIR / f"{country}_{level}_units.csv"


def save_levels(country, base, levels):
    """Dissolve base polygons per level and save one units table per level (skip if it exists)."""
    for level in levels:
        f = out_path(country, level)
        if f.exists():
            print("Exists, skipped:", f)
            continue
        dis = base.dissolve(by=level, aggfunc={POP_COL: "sum", AREA_COL: "sum"})
        cen = dis.geometry.centroid
        units = pd.DataFrame({"unit_id": dis.index.astype(str), "population": dis[POP_COL].to_numpy(),
                              "area_km2": dis[AREA_COL].to_numpy(),
                              "x_m": cen.x.to_numpy(), "y_m": cen.y.to_numpy()}).sort_values("unit_id")
        units.to_csv(f, index=False)
        print("Saved:", f, "| units:", len(units))


# --- CZ: obec -> orp -> okres -> kraj mapping from the OD file, geometry from the raw GISCO file ---
od = pd.read_parquet(CZ_OD)
cols = ["obec", "orp", "okres"]
orig = od[["op_obec_kod", "op_orp_kod", "op_lau1_kod"]].set_axis(cols, axis=1)
dest = od[["doj_obec_kod", "doj_orp_kod", "doj_lau1_kod"]].set_axis(cols, axis=1)
cz_map = pd.concat([orig, dest]).astype(str).drop_duplicates()
cz_map["kraj"] = cz_map["okres"].str[:CZ_KRAJ_PREFIX_LEN]

cz = pd.read_parquet(CZ_INPUTS, columns=["LAU_ID", POP_COL, AREA_COL])
cz["obec"] = cz["LAU_ID"].astype(str)
geom = gpd.read_file(GEOM, where="CNTR_CODE = 'CZ'")
geom["obec"] = geom["LAU_ID"].astype(str)
cz = cz.merge(cz_map, on="obec", how="left").merge(geom[["obec", "geometry"]], on="obec", how="left")
assert cz[["okres", "geometry"]].notna().all().all(), "CZ obec without mapping or geometry"
save_levels("CZ", gpd.GeoDataFrame(cz, geometry="geometry", crs=CRS_METRIC), CZ_LEVELS)

# --- PL: gmina -> powiat -> nuts3 mapping, geometry from the raw GISCO file ---
pl = pd.read_parquet(PL_INPUTS, columns=["p6", "LAU_ID", POP_COL, AREA_COL])
pl["gmina"] = pl["p6"].astype(str).str.zfill(PL_CODE_LEN)
pl["LAU_ID"] = pl["LAU_ID"].astype(str)
pl_map = pd.read_csv(PL_MAP, dtype=str)[["gmina_code", "powiat_code", "nuts3_code"]]
pl_map.columns = ["gmina", "powiat", "nuts3"]
geom = gpd.read_file(GEOM, where="CNTR_CODE = 'PL'")
geom["LAU_ID"] = geom["LAU_ID"].astype(str)
pl = pl.merge(pl_map, on="gmina", how="left").merge(geom[["LAU_ID", "geometry"]], on="LAU_ID", how="left")
assert pl[["powiat", "geometry"]].notna().all().all(), "PL gmina without mapping or geometry"
save_levels("PL", gpd.GeoDataFrame(pl, geometry="geometry", crs=CRS_METRIC), PL_LEVELS)

# --- check: distances from the new centroids vs the 06 / 06b distance tables ---
for (country, level), ref in REF.items():
    u = pd.read_csv(out_path(country, level), dtype={"unit_id": str}).set_index("unit_id")
    r = pd.read_csv(ref, index_col=0)
    r.index, r.columns = r.index.astype(str), r.columns.astype(str)
    common = [i for i in u.index if i in r.index]
    d = squareform(pdist(u.loc[common, ["x_m", "y_m"]].to_numpy())) / 1000
    diff = np.abs(d - r.loc[common, common].to_numpy())
    print(country, level, "| units:", len(u), "| matched ids:", len(common),
          "| max abs distance difference (km):", round(float(diff.max()), 2))

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
OUT_DIR = Path("data/interim/09_lenormand_multilevel")
LEVELS = {"CZ": ["obec", "orp", "okres", "kraj"], "PL": ["gmina", "powiat", "nuts3"]}
DTYPE = np.float32
# ------------------------------------------------------------------

for country, levels in LEVELS.items():
    for level in levels:
        units_file = OUT_DIR / f"{country}_{level}_units.csv"
        dist_file = OUT_DIR / f"{country}_{level}_distance_km.npy"
        if dist_file.exists():
            print("Exists, skipped:", dist_file)
            continue
        u = pd.read_csv(units_file, dtype={"unit_id": str})
        # Euclidean distance between centroids (metres -> km), computed in float64 first
        d = squareform(pdist(u[["x_m", "y_m"]].to_numpy())) / 1000
        np.save(dist_file, d.astype(DTYPE))
        off = d[~np.eye(len(d), dtype=bool)]
        print(country, level, "| units:", len(u),
              "| symmetric:", bool(np.allclose(d, d.T)),
              "| zero diagonal:", bool(np.allclose(np.diag(d), 0)),
              "| min/max km:", round(off.min(), 2), round(off.max(), 1))
        del d, off

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
INTERIM = Path("data/interim")
OUT_DIR = INTERIM / "09_lenormand_multilevel"
CZ_LEVELS = ["obec", "orp", "okres", "kraj"]
PL_LEVELS = ["gmina", "powiat", "nuts3"]
CZ_KRAJ_PREFIX_LEN = 5            # kraj = first 5 characters of the okres code
PL_CODE_LEN = 6                   # gmina code length (leading zeros)
CZ_OD = INTERIM / "cz_od_work_obec.parquet"
PL_OD = INTERIM / "pl_od_work_gmina.parquet"
PL_MAP = INTERIM / "pl_gmina_nuts3.csv"
DTYPE = np.float32
# reference OD matrices of notebooks 06 / 06b (used only for the check)
REF = {("CZ", "kraj"): INTERIM / "lenormand_comparison/cz/od_observed.csv",
       ("PL", "nuts3"): INTERIM / "lenormand_comparison/pl/od_observed_missing0.csv"}
# ------------------------------------------------------------------
od_file = lambda country, level: OUT_DIR / f"{country}_{level}_od_observed.npy"
units_file = lambda country, level: OUT_DIR / f"{country}_{level}_units.csv"


def build_matrix(flows, country, level):
    """Sum flows per (origin, destination) unit, place them in a dense matrix, zero the diagonal."""
    f = od_file(country, level)
    if f.exists():
        print("Exists, skipped:", f)
        return
    ids = pd.read_csv(units_file(country, level), dtype={"unit_id": str})["unit_id"]
    pos = pd.Series(np.arange(len(ids)), index=ids)
    g = flows.groupby(["o", "d"], as_index=False)["w"].sum()
    i, j = g["o"].map(pos), g["d"].map(pos)
    assert i.notna().all() and j.notna().all(), "flow endpoint without a unit"
    m = np.zeros((len(ids), len(ids)), dtype=DTYPE)
    m[i.astype(int), j.astype(int)] = g["w"].to_numpy()
    intra = float(np.trace(m))
    np.fill_diagonal(m, 0)
    np.save(f, m)
    print(country, level, "| units:", len(ids), "| nonzero pairs:", int((m > 0).sum()),
          "| total flow (off-diagonal):", int(m.sum()), "| intra-unit flow dropped:", int(intra))


# --- CZ: obec -> orp -> okres -> kraj mapping from the OD file itself ---
cz = pd.read_parquet(CZ_OD)
cols = ["obec", "orp", "okres"]
orig = cz[["op_obec_kod", "op_orp_kod", "op_lau1_kod"]].set_axis(cols, axis=1)
dest = cz[["doj_obec_kod", "doj_orp_kod", "doj_lau1_kod"]].set_axis(cols, axis=1)
cz_map = pd.concat([orig, dest]).astype(str).drop_duplicates().set_index("obec")
cz_map["kraj"] = cz_map["okres"].str[:CZ_KRAJ_PREFIX_LEN]
assert cz_map.index.is_unique, "an obec maps to more than one orp/okres"
o_obec, d_obec = cz["op_obec_kod"].astype(str), cz["doj_obec_kod"].astype(str)
for level in CZ_LEVELS:
    key = (lambda s: s) if level == "obec" else (lambda s, lv=level: s.map(cz_map[lv]))
    flows = pd.DataFrame({"o": key(o_obec), "d": key(d_obec), "w": cz["dojizdka_prace"]})
    build_matrix(flows, "CZ", level)

# --- PL: gmina -> powiat -> nuts3 mapping ---
pl = pd.read_parquet(PL_OD)
pl_map = pd.read_csv(PL_MAP, dtype=str)[["gmina_code", "powiat_code", "nuts3_code"]]
pl_map.columns = ["gmina", "powiat", "nuts3"]
pl_map = pl_map.set_index("gmina")
o_g = pl["p6_r"].astype(str).str.zfill(PL_CODE_LEN)
d_g = pl["p6_w"].astype(str).str.zfill(PL_CODE_LEN)
print("PL OD gminas without mapping:", int((~o_g.isin(pl_map.index)).sum() + (~d_g.isin(pl_map.index)).sum()))
for level in PL_LEVELS:
    key = (lambda s: s) if level == "gmina" else (lambda s, lv=level: s.map(pl_map[lv]))
    flows = pd.DataFrame({"o": key(o_g), "d": key(d_g), "w": pl["flow"]})
    build_matrix(flows, "PL", level)

# --- check against the matrices of notebooks 06 / 06b ---
for (country, level), ref in REF.items():
    ids = pd.read_csv(units_file(country, level), dtype={"unit_id": str})["unit_id"].tolist()
    ours = pd.DataFrame(np.load(od_file(country, level)), index=ids, columns=ids)
    r = pd.read_csv(ref, index_col=0)
    r.index, r.columns = r.index.astype(str), r.columns.astype(str)
    common = [i for i in ids if i in r.index and i in r.columns]
    diff = np.abs(ours.loc[common, common].to_numpy() - r.loc[common, common].to_numpy())
    print(country, level, "| matched ids:", len(common), "of", len(ids),
          "| our total:", int(ours.to_numpy().sum()), "| reference total:", int(np.nansum(r.to_numpy())),
          "| max abs difference:", float(diff.max()))

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
IN_DIR = Path("data/interim/09_lenormand_multilevel")
OUT_DIR = Path("data/processed/09_lenormand_multilevel")
OUT_FILE = OUT_DIR / "distance_stats.csv"
LEVELS = {"CZ": ["obec", "orp", "okres", "kraj"], "PL": ["gmina", "powiat", "nuts3"]}
# ------------------------------------------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)


def weighted_stats(w, d):
    """Flow-weighted mean and Pearson kurtosis (m4 / m2^2, not excess) of the distances."""
    w, d = w.astype(np.float64), d.astype(np.float64)
    total = w.sum()
    mean = (w * d).sum() / total
    m2 = (w * (d - mean) ** 2).sum() / total
    m4 = (w * (d - mean) ** 4).sum() / total
    return total, mean, m4 / m2 ** 2


rows = []
for country, levels in LEVELS.items():
    for level in levels:
        od = np.load(IN_DIR / f"{country}_{level}_od_observed.npy")
        dist = np.load(IN_DIR / f"{country}_{level}_distance_km.npy")
        units = pd.read_csv(IN_DIR / f"{country}_{level}_units.csv")
        mask = od > 0                       # the diagonal is zero, so it is excluded too
        total, mean_d, kurt = weighted_stats(od[mask], dist[mask])
        rows.append({"country": country, "level": level, "n_units": len(units),
                     "mean_unit_area_km2": units["area_km2"].mean(),
                     "total_flow": total, "mean_distance_km": mean_d,
                     "kurtosis_pearson": kurt})
        del od, dist, mask

stats = pd.DataFrame(rows)
stats.to_csv(OUT_FILE, index=False)
print(stats.round(2).to_string(index=False))
print("saved:", OUT_FILE)

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
IN_DIR = Path("data/interim/09_lenormand_multilevel")
OUT_DIR = Path("data/processed/09_lenormand_multilevel")
OUT_FILE = OUT_DIR / "distance_stats_variants.csv"
LEVELS = {"CZ": ["obec", "orp", "okres", "kraj"], "PL": ["gmina", "powiat", "nuts3"]}
BIN_KM = 2.0                      # bin width of the paper's CPCd measure
# ------------------------------------------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)


def mean_kurtosis(x, w=None):
    """(Weighted) mean and Pearson kurtosis (m4 / m2^2, not excess) of x."""
    x = x.astype(np.float64)
    if w is None:
        w = np.ones_like(x)
    else:
        w = w.astype(np.float64)
    total = w.sum()
    mean = (w * x).sum() / total
    m2 = (w * (x - mean) ** 2).sum() / total
    m4 = (w * (x - mean) ** 4).sum() / total
    return mean, m4 / m2 ** 2


rows = []
for country, levels in LEVELS.items():
    for level in levels:
        od = np.load(IN_DIR / f"{country}_{level}_od_observed.npy")
        dist = np.load(IN_DIR / f"{country}_{level}_distance_km.npy")
        area = pd.read_csv(IN_DIR / f"{country}_{level}_units.csv")["area_km2"]
        obs = od > 0                                  # observed pairs (diagonal is zero)
        d_obs, w_obs = dist[obs], od[obs]
        # variant 4: flow-weighted histogram in BIN_KM-wide bins, bin centres as values
        idx = np.floor(d_obs / BIN_KM).astype(np.int64)
        counts = np.bincount(idx, weights=w_obs.astype(np.float64))
        centres = (np.arange(len(counts)) + 0.5) * BIN_KM
        off_diag = ~np.eye(len(dist), dtype=bool)     # all unit pairs i != j
        variants = {"flow_weighted": mean_kurtosis(d_obs, w_obs),
                    "pairs_unweighted_observed": mean_kurtosis(d_obs),
                    "all_pairs_unweighted": mean_kurtosis(dist[off_diag]),
                    "flow_weighted_binned": mean_kurtosis(centres, counts)}
        for name, (mean_d, kurt) in variants.items():
            rows.append({"country": country, "level": level, "n_units": len(area),
                         "mean_unit_area_km2": round(area.mean(), 1), "variant": name,
                         "mean_distance_km": mean_d, "kurtosis_pearson": kurt})
        del od, dist, obs, off_diag

res = pd.DataFrame(rows)
res.to_csv(OUT_FILE, index=False)
wide = res.pivot_table(index=["country", "level", "n_units", "mean_unit_area_km2"],
                       columns="variant", values="kurtosis_pearson", sort=False).round(2)
print(wide.to_string())
print("saved:", OUT_FILE)

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, ScalarFormatter

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
DATA_DIR = Path("data/processed/09_lenormand_multilevel")
IN_FILE = DATA_DIR / "distance_stats.csv"
FIG_DIR = Path("figures")                # working figures; article-ready ones go to plots/
FIG_NAME = "fig9_distance_stats.png"
X_COL = "mean_unit_area_km2"
# (column, y label, panel tag, position of the fit text in axes coordinates)
PANELS = [("mean_distance_km", "Average commuting distance (km)", "(a)", (0.50, 0.06)),
          ("kurtosis_pearson", "Kurtosis (Pearson)", "(b)", (0.05, 0.06))]
COLORS = {"CZ": "tab:blue", "PL": "tab:red"}
X_SUBS = (1.0, 2.0, 5.0)                 # labelled ticks per decade, x axis
Y_SUBS = (1.0, 2.0, 3.0, 5.0, 7.0)       # labelled ticks per decade, y axis
FIG_SIZE = (11, 4.5)
DPI = 150
# ------------------------------------------------------------------


def plain_log_axis(axis, subs):
    """Log axis with plain-number tick labels (no 10^n form), also on the minor ticks."""
    axis.set_major_locator(LogLocator(base=10, subs=(1.0,), numticks=20))
    axis.set_minor_locator(LogLocator(base=10, subs=[s for s in subs if s != 1.0], numticks=20))
    for set_formatter in (axis.set_major_formatter, axis.set_minor_formatter):
        fmt = ScalarFormatter()
        fmt.set_scientific(False)
        set_formatter(fmt)


stats = pd.read_csv(IN_FILE)

fig, axes = plt.subplots(1, len(PANELS), figsize=FIG_SIZE)
for ax, (col, ylabel, tag, text_xy) in zip(axes, PANELS):
    for country, g in stats.groupby("country", sort=False):
        # one point per (country, level) case study, no connecting line
        ax.plot(g[X_COL], g[col], "o", color=COLORS.get(country), label=country)
        for _, r in g.iterrows():
            ax.annotate(r["level"], (r[X_COL], r[col]), textcoords="offset points",
                        xytext=(4, 4), fontsize=8)
    # power-law fit over all points = straight line in log-log space
    lx, ly = np.log10(stats[X_COL]), np.log10(stats[col])
    slope, intercept = np.polyfit(lx, ly, 1)
    r2 = np.corrcoef(lx, ly)[0, 1] ** 2
    xs = np.linspace(lx.min(), lx.max(), 100)
    ax.plot(10 ** xs, 10 ** (intercept + slope * xs), "-", color="black", lw=1)
    ax.text(*text_xy, f"y = {10 ** intercept:.3g} · <S>$^{{{slope:.2f}}}$\n$R^2$ = {r2:.2f}",
            transform=ax.transAxes, fontsize=9)
    ax.set_xscale("log")
    ax.set_yscale("log")
    plain_log_axis(ax.xaxis, X_SUBS)
    plain_log_axis(ax.yaxis, Y_SUBS)
    ax.tick_params(which="both", labelsize=9)
    ax.set_xlabel("<S> (km$^2$)")
    ax.set_ylabel(ylabel)
    ax.set_title(tag, loc="left", fontweight="bold")
    ax.legend(frameon=False)
    print(tag, "| slope:", round(slope, 3), "| R2:", round(r2, 3))

fig.tight_layout()
FIG_DIR.mkdir(exist_ok=True)
fig.savefig(FIG_DIR / FIG_NAME, dpi=DPI)
plt.show()
print("saved:", FIG_DIR / FIG_NAME)

In [ ]:
import io
import os
import gc
import time
import warnings
from contextlib import redirect_stdout, redirect_stderr
from pathlib import Path
import numpy as np
import pandas as pd
from TDLM import tdlm

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
IN_DIR = Path("data/interim/09_lenormand_multilevel")
CALIB_DIR = IN_DIR / "calibration_average"
OUT_DIR = Path("data/processed/09_lenormand_multilevel")
OUT_FILE = OUT_DIR / "calibrated_exponents_levels.csv"
ALL_LEVELS = [("CZ", "kraj"), ("PL", "nuts3"), ("CZ", "okres"), ("PL", "powiat"),
              ("CZ", "orp"), ("PL", "gmina"), ("CZ", "obec")]      # small to large
OVERWRITE_LEVELS = []                             # recomputed even if the checkpoint file exists
CALIB_LAWS = ["NGravExp", "NGravPow"]             # Rad has no parameter
CONSTRAINT_LEVELS = ["UM", "PCM", "ACM", "DCM"]
AVERAGE = True                                    # deterministic average matrix instead of random replications
N_REPS_CALIB = 10                                 # ignored by the library when AVERAGE is True
PROCESSES = None                                  # None = library default (CPU count - 2)
SEED = 0
CALIB_MEASURE = "CPC"
# reference files of notebooks 07 / 07b (used only for the check)
REF = {("CZ", "kraj"): Path("data/processed/07_lenormand_pytdlm_cz/calibrated_exponents.csv"),
       ("PL", "nuts3"): Path("data/processed/07b_lenormand_pytdlm_pl/calibrated_exponents.csv")}
# ------------------------------------------------------------------
CALIB_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=RuntimeWarning)
calib_path = lambda country, level: CALIB_DIR / f"{country}_{level}_calibrated_exponents.csv"

for country, level in ALL_LEVELS:
    f = calib_path(country, level)
    if f.exists() and (country, level) not in OVERWRITE_LEVELS:
        print("Exists, skipped:", f)
        continue
    if f.exists():
        print("Overwriting:", f)
    units = pd.read_csv(IN_DIR / f"{country}_{level}_units.csv")
    mass = units["population"].to_numpy(dtype=np.float64)
    distance = np.load(IN_DIR / f"{country}_{level}_distance_km.npy").astype(np.float64)
    obs = np.load(IN_DIR / f"{country}_{level}_od_observed.npy").astype(np.float64)
    out_trips, in_trips = obs.sum(axis=1), obs.sum(axis=0)
    t0 = time.time()
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        calib = tdlm.run_optimization(
            mass_origin=mass, mass_destination=mass, distance=distance, obs=obs,
            law=CALIB_LAWS, model=CONSTRAINT_LEVELS,
            out_trips=out_trips, in_trips=in_trips,
            average=AVERAGE, repli=N_REPS_CALIB, measure=CALIB_MEASURE,
            processes=PROCESSES, random_seed=SEED)
    calib.insert(0, "level", level)
    calib.insert(0, "country", country)
    calib["n_units"] = len(units)
    calib["mean_unit_area_km2"] = units["area_km2"].mean()
    calib.to_csv(f, index=False)
    print(f"Saved: {f} | units: {len(units)} | {time.time() - t0:.1f} s")
    del mass, distance, obs, out_trips, in_trips, calib
    gc.collect()

# --- combine all finished levels into one table ---
done = [pd.read_csv(calib_path(c, l)) for c, l in ALL_LEVELS if calib_path(c, l).exists()]
table = pd.concat(done, ignore_index=True)
table.to_csv(OUT_FILE, index=False)
print("saved:", OUT_FILE, "| rows:", len(table))
print(table.pivot_table(index=["country", "level", "mean_unit_area_km2"], columns=["Law", "Model"],
                        values="Exponent", sort=False).round(4).to_string())

# --- check against notebooks 07 / 07b (replication-based calibration) ---
for (country, level), ref in REF.items():
    if not (ref.exists() and calib_path(country, level).exists()):
        continue
    r = pd.read_csv(ref)
    if "Variant" in r.columns:
        r = r[r["Variant"] == "missing0"]
    ours = pd.read_csv(calib_path(country, level))
    m = ours.merge(r[["Law", "Model", "Exponent"]], on=["Law", "Model"], suffixes=("", "_ref"))
    print(country, level, "| matched rows:", len(m),
          "| max abs difference vs reference:", float((m["Exponent"] - m["Exponent_ref"]).abs().max()))

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, ScalarFormatter

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
IN_FILE = Path("data/processed/09_lenormand_multilevel/calibrated_exponents_levels.csv")
FIG_DIR = Path("figures")                # working figures; article-ready ones go to plots/
FIG_NAME = "fig8_parameter_vs_area.png"
X_COL = "mean_unit_area_km2"
# (law, y label, panel tag, minor tick multipliers on the y axis, position of the fit text)
PANELS = [("NGravExp", "β (exponential distance decay)", "(a)", (1.0, 2.0, 3.0, 5.0), (0.05, 0.06)),
          ("NGravPow", "β (power distance decay)", "(b)", (1.0, 1.5, 2.0, 3.0, 4.0), (0.05, 0.06))]
CONSTRAINT_ORDER = ["UM", "PCM", "ACM", "DCM"]
SHADES = {"UM": "0.0", "PCM": "0.35", "ACM": "0.55", "DCM": "0.75"}   # dark to light grey
MARKERS = {"CZ": "o", "PL": "s"}
X_SUBS = (1.0, 2.0, 5.0)                 # labelled ticks per decade, x axis
FIG_SIZE = (11, 4.5)
DPI = 150
# ------------------------------------------------------------------


def plain_log_axis(axis, subs):
    """Log axis with plain-number tick labels (no 10^n form), also on the minor ticks."""
    axis.set_major_locator(LogLocator(base=10, subs=(1.0,), numticks=20))
    axis.set_minor_locator(LogLocator(base=10, subs=[s for s in subs if s != 1.0], numticks=20))
    for set_formatter in (axis.set_major_formatter, axis.set_minor_formatter):
        fmt = ScalarFormatter()
        fmt.set_scientific(False)
        set_formatter(fmt)


table = pd.read_csv(IN_FILE)
print("levels in the table:", sorted(set(zip(table["country"], table["level"]))))

fig, axes = plt.subplots(1, len(PANELS), figsize=FIG_SIZE)
for ax, (law, ylabel, tag, y_subs, text_xy) in zip(axes, PANELS):
    d = table[table["Law"] == law]
    for country, marker in MARKERS.items():
        for model in CONSTRAINT_ORDER:
            g = d[(d["country"] == country) & (d["Model"] == model)]
            ax.scatter(g[X_COL], g["Exponent"], marker=marker, s=40, color=SHADES[model],
                       edgecolor="black", linewidth=0.5, zorder=3)
    # power-law fit over all points of the panel = straight line in log-log space
    lx, ly = np.log10(d[X_COL]), np.log10(d["Exponent"])
    slope, intercept = np.polyfit(lx, ly, 1)
    r2 = np.corrcoef(lx, ly)[0, 1] ** 2
    xs = np.linspace(lx.min(), lx.max(), 100)
    ax.plot(10 ** xs, 10 ** (intercept + slope * xs), "-", color="tab:red", lw=1.2, zorder=2)
    ax.text(*text_xy, f"β = {10 ** intercept:.3g} · <S>$^{{{slope:.2f}}}$\n$R^2$ = {r2:.2f}",
            transform=ax.transAxes, fontsize=9)
    ax.set_xscale("log")
    ax.set_yscale("log")
    plain_log_axis(ax.xaxis, X_SUBS)
    plain_log_axis(ax.yaxis, y_subs)
    ax.tick_params(which="both", labelsize=9)
    ax.set_xlabel("<S> (km$^2$)")
    ax.set_ylabel(ylabel)
    ax.set_title(tag, loc="left", fontweight="bold")
    print(tag, law, "| slope:", round(slope, 3), "| R2:", round(r2, 3), "| points:", len(d))

# legend: grey shade = constraint level, marker = country
handles = [plt.Line2D([], [], marker="o", linestyle="", color=SHADES[m], markeredgecolor="black",
                      label=m) for m in CONSTRAINT_ORDER]
handles += [plt.Line2D([], [], marker=mk, linestyle="", color="white", markeredgecolor="black",
                       label=c) for c, mk in MARKERS.items()]
axes[-1].legend(handles=handles, frameon=False, fontsize=8, loc="upper left")

fig.tight_layout()
FIG_DIR.mkdir(exist_ok=True)
fig.savefig(FIG_DIR / FIG_NAME, dpi=DPI)
plt.show()
print("saved:", FIG_DIR / FIG_NAME)

In [ ]:
import io
import os
import gc
import time
import warnings
from contextlib import redirect_stdout, redirect_stderr
from pathlib import Path
import numpy as np
import pandas as pd
from TDLM import tdlm

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
IN_DIR = Path("data/interim/09_lenormand_multilevel")
CPC_DIR = IN_DIR / "fig10_cpc"
CALIB_FILE = Path("data/processed/09_lenormand_multilevel/calibrated_exponents_levels.csv")
OUT_FILE = Path("data/processed/09_lenormand_multilevel/cpc_percentage_error_population.csv")
LAWS = ["NGravExp", "NGravPow"]
X_COL, EXP_COL, CPC_COL = "mean_unit_area_km2", "Exponent", "CPC"
AVERAGE = True                       # same deterministic average matrix as in the calibration
PROCESSES = None                     # None = library default (CPU count - 2)
SEED = 0
# ------------------------------------------------------------------
CPC_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=RuntimeWarning)
first_value = lambda d: next(iter(d.values())) if isinstance(d, dict) else d

table = pd.read_csv(CALIB_FILE)

# --- 1) one power-law fit per law over all points (the Fig. 8 lines) ---
fits = {}
for law in LAWS:
    d = table[table["Law"] == law]
    slope, intercept = np.polyfit(np.log10(d[X_COL]), np.log10(d[EXP_COL]), 1)
    fits[law] = (slope, intercept)
    print(law, "| slope:", round(slope, 3), "| prefactor:", round(10 ** intercept, 4))
table["Exponent_est"] = [10 ** (fits[l][1] + fits[l][0] * np.log10(s))
                         for l, s in zip(table["Law"], table[X_COL])]


def simulate_cpc(inp, law, model, exponent):
    """CPC of one law / constraint level with a given exponent (average matrix)."""
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        sim = tdlm.run_law_model(
            law=law, mass_origin=inp["mass"], mass_destination=inp["mass"],
            distance=inp["distance"], exponent=np.array([exponent]), model=model,
            out_trips=inp["out_trips"], in_trips=inp["in_trips"],
            average=AVERAGE, repli=1, processes=PROCESSES, random_seed=SEED, verbose=False)
        res = tdlm.gof(sim=sim, obs=inp["obs"], distance=inp["distance"],
                       measures=["CPC"], processes=PROCESSES, verbose=False)
    return float(first_value(res)["CPC"].mean())


# --- 2) CPC with the calibrated and with the estimated exponent, one checkpoint per level ---
for country, level in table[["country", "level"]].drop_duplicates().itertuples(index=False):
    f = CPC_DIR / f"{country}_{level}_cpc.csv"
    if f.exists():
        print("Exists, skipped:", f)
        continue
    t0 = time.time()
    units = pd.read_csv(IN_DIR / f"{country}_{level}_units.csv")
    obs = np.load(IN_DIR / f"{country}_{level}_od_observed.npy").astype(np.float64)
    inp = {"mass": units["population"].to_numpy(dtype=np.float64),
           "distance": np.load(IN_DIR / f"{country}_{level}_distance_km.npy").astype(np.float64),
           "obs": obs, "out_trips": obs.sum(axis=1), "in_trips": obs.sum(axis=0)}
    rows = table[(table["country"] == country) & (table["level"] == level)].copy()
    rows["CPC_calibrated_recomputed"] = [simulate_cpc(inp, r.Law, r.Model, r.Exponent)
                                         for r in rows.itertuples()]
    rows["CPC_estimated"] = [simulate_cpc(inp, r.Law, r.Model, r.Exponent_est)
                             for r in rows.itertuples()]
    rows.to_csv(f, index=False)
    print(f"Saved: {f} | {time.time() - t0:.1f} s")
    del inp, obs
    gc.collect()

# --- 3) combine, check, percentage error ---
res = pd.concat([pd.read_csv(CPC_DIR / f"{c}_{l}_cpc.csv")
                 for c, l in table[["country", "level"]].drop_duplicates().itertuples(index=False)],
                ignore_index=True)
res["abs_pct_error"] = 100 * (res["CPC_estimated"] - res[CPC_COL]).abs() / res[CPC_COL]
res.to_csv(OUT_FILE, index=False)
print("saved:", OUT_FILE, "| rows:", len(res))
print("check, max abs difference between the recomputed and the saved calibrated CPC:",
      float((res["CPC_calibrated_recomputed"] - res[CPC_COL]).abs().max()))
print(res.groupby("Law")["abs_pct_error"].describe()[["count", "min", "50%", "max"]].round(2).to_string())
print(res.pivot_table(index=["country", "level"], columns=["Law", "Model"],
                      values="abs_pct_error", sort=False).round(1).to_string())

In [ ]:
import io
import os
import gc
import time
import warnings
from contextlib import redirect_stdout, redirect_stderr
from pathlib import Path
import numpy as np
import pandas as pd
from TDLM import tdlm

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
IN_DIR = Path("data/interim/09_lenormand_multilevel")
FLOWS_DIR = IN_DIR / "fig10_flows"
POP_CALIB_FILE = Path("data/processed/09_lenormand_multilevel/calibrated_exponents_levels.csv")
OUT_FILE = Path("data/processed/09_lenormand_multilevel/cpc_percentage_error_flows.csv")
ALL_LEVELS = [("CZ", "kraj"), ("PL", "nuts3"), ("CZ", "okres"), ("PL", "powiat"),
              ("CZ", "orp"), ("PL", "gmina"), ("CZ", "obec")]      # small to large
LAWS = ["NGravExp", "NGravPow"]
CONSTRAINT_LEVELS = ["UM", "PCM", "ACM", "DCM"]
X_COL, EXP_COL = "mean_unit_area_km2", "Exponent"
AVERAGE = True                       # same deterministic average matrix as in the population-based runs
N_REPS_CALIB = 10                    # ignored by the library when AVERAGE is True
CALIB_MEASURE = "CPC"
PROCESSES = None                     # None = library default (CPU count - 2)
SEED = 0
# ------------------------------------------------------------------
FLOWS_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=RuntimeWarning)
first_value = lambda d: next(iter(d.values())) if isinstance(d, dict) else d

# --- power-law fits of the population-based exponents (the Fig. 8 lines), used for the estimates ---
pop = pd.read_csv(POP_CALIB_FILE)
fits = {}
for law in LAWS:
    d = pop[pop["Law"] == law]
    fits[law] = np.polyfit(np.log10(d[X_COL]), np.log10(d[EXP_COL]), 1)
    print(law, "| slope:", round(fits[law][0], 3), "| prefactor:", round(10 ** fits[law][1], 4))


def simulate_cpc(inp, law, model, exponent):
    """CPC of one law / constraint level with a given exponent; the masses are O_i and D_j."""
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        sim = tdlm.run_law_model(
            law=law, mass_origin=inp["out_trips"], mass_destination=inp["in_trips"],
            distance=inp["distance"], exponent=np.array([exponent]), model=model,
            out_trips=inp["out_trips"], in_trips=inp["in_trips"],
            average=AVERAGE, repli=1, processes=PROCESSES, random_seed=SEED, verbose=False)
        res = tdlm.gof(sim=sim, obs=inp["obs"], distance=inp["distance"],
                       measures=["CPC"], processes=PROCESSES, verbose=False)
    return float(first_value(res)["CPC"].mean())


for country, level in ALL_LEVELS:
    f = FLOWS_DIR / f"{country}_{level}_flows.csv"
    if f.exists():
        print("Exists, skipped:", f)
        continue
    t0 = time.time()
    units = pd.read_csv(IN_DIR / f"{country}_{level}_units.csv")
    obs = np.load(IN_DIR / f"{country}_{level}_od_observed.npy").astype(np.float64)
    inp = {"distance": np.load(IN_DIR / f"{country}_{level}_distance_km.npy").astype(np.float64),
           "obs": obs, "out_trips": obs.sum(axis=1), "in_trips": obs.sum(axis=0)}
    # 1) calibration with O_i and D_j as masses
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        calib = tdlm.run_optimization(
            mass_origin=inp["out_trips"], mass_destination=inp["in_trips"],
            distance=inp["distance"], obs=obs, law=LAWS, model=CONSTRAINT_LEVELS,
            out_trips=inp["out_trips"], in_trips=inp["in_trips"],
            average=AVERAGE, repli=N_REPS_CALIB, measure=CALIB_MEASURE,
            processes=PROCESSES, random_seed=SEED)
    mean_area = units["area_km2"].mean()
    calib.insert(0, "level", level)
    calib.insert(0, "country", country)
    calib["mean_unit_area_km2"] = mean_area
    # 2) exponent estimated from the population-based regression at this level's <S>
    calib["Exponent_est"] = [10 ** (fits[l][1] + fits[l][0] * np.log10(mean_area)) for l in calib["Law"]]
    # 3) CPC with the calibrated (recomputed) and with the estimated exponent
    calib["CPC_calibrated_recomputed"] = [simulate_cpc(inp, r.Law, r.Model, r.Exponent)
                                          for r in calib.itertuples()]
    calib["CPC_estimated"] = [simulate_cpc(inp, r.Law, r.Model, r.Exponent_est)
                              for r in calib.itertuples()]
    calib.to_csv(f, index=False)
    print(f"Saved: {f} | units: {len(units)} | {time.time() - t0:.1f} s")
    del inp, obs, calib
    gc.collect()

# --- combine, check, percentage error ---
res = pd.concat([pd.read_csv(FLOWS_DIR / f"{c}_{l}_flows.csv") for c, l in ALL_LEVELS], ignore_index=True)
res["abs_pct_error"] = 100 * (res["CPC_estimated"] - res["CPC"]).abs() / res["CPC"]
res.to_csv(OUT_FILE, index=False)
print("saved:", OUT_FILE, "| rows:", len(res))
print("check, max abs difference between the recomputed and the saved calibrated CPC:",
      float((res["CPC_calibrated_recomputed"] - res["CPC"]).abs().max()))
print(res.groupby("Law")["abs_pct_error"].describe()[["count", "min", "50%", "max"]].round(2).to_string())
print(res.pivot_table(index=["country", "level"], columns=["Law", "Model"],
                      values="abs_pct_error", sort=False).round(1).to_string())

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
DATA_DIR = Path("data/processed/09_lenormand_multilevel")
POP_FILE = DATA_DIR / "cpc_percentage_error_population.csv"
FLOWS_FILE = DATA_DIR / "cpc_percentage_error_flows.csv"
SUMMARY_FILE = DATA_DIR / "fig10_error_summary.csv"
FIG_DIR = Path("figures")                # working figures; article-ready ones go to plots/
FIG_NAME = "fig10_cpc_percentage_error.png"
ERR_COL = "abs_pct_error"
LAWS = [("NGravExp", "NGrav (exp)", "tab:red"), ("NGravPow", "NGrav (pow)", "tab:blue")]
WHISKERS = (10, 90)                      # first and last decile, as in the paper
BOX_WIDTH = 0.32
BOX_OFFSET = 0.2                         # distance of the two boxes of a law from the group centre
GROUP_SPACING = 1.2
FIG_SIZE = (7, 4.5)
DPI = 150
# ------------------------------------------------------------------

pop = pd.read_csv(POP_FILE)
flows = pd.read_csv(FLOWS_FILE)

fig, ax = plt.subplots(figsize=FIG_SIZE)
tick_pos, tick_lab, rows = [], [], []
for i, (law, label, color) in enumerate(LAWS):
    centre = i * GROUP_SPACING
    for source, df, offset, notch in [("population", pop, -BOX_OFFSET, True),
                                      ("in/out flows", flows, BOX_OFFSET, False)]:
        vals = df.loc[df["Law"] == law, ERR_COL].to_numpy()
        ax.boxplot(vals, positions=[centre + offset], widths=BOX_WIDTH, notch=notch,
                   whis=WHISKERS, patch_artist=True,
                   boxprops=dict(facecolor=color, alpha=0.6, edgecolor="black"),
                   medianprops=dict(color="black", linewidth=1.5),
                   flierprops=dict(marker="o", markersize=3, markerfacecolor=color, markeredgecolor=color))
        tick_pos.append(centre + offset)
        tick_lab.append(source)
        rows.append({"law": law, "input": source, "n": len(vals), "median_pct": float(pd.Series(vals).median()),
                     "max_pct": float(vals.max())})
    ax.text(centre, -0.17, label, transform=ax.get_xaxis_transform(), ha="center", fontsize=10)

ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_lab, fontsize=8)
ax.set_ylabel("CPC percentage error")
ax.set_ylim(bottom=-0.5)
ax.set_title("Notched box: population | classic box: in/out flows", fontsize=9, loc="left")

fig.tight_layout()
FIG_DIR.mkdir(exist_ok=True)
fig.savefig(FIG_DIR / FIG_NAME, dpi=DPI, bbox_inches="tight")
plt.show()
print("saved:", FIG_DIR / FIG_NAME)

summary = pd.DataFrame(rows)
summary.to_csv(SUMMARY_FILE, index=False)
print("saved:", SUMMARY_FILE)
print(summary.round(2).to_string(index=False))